# Week 6 guided practice
CML3111. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and keep its files together. For file-reading examples, upload the supplied CSV into the Colab runtime before running the file-reading example. Small worked datasets are invented; the CSV inspection uses your supplied file.


## Today’s learning
*PowerPoint slide 2*

Use pandas tables for this week’s data work. Explain the decisions and check the outputs.

Complete the issued function interfaces and written evidence using the demonstrated workflow.




## From Week 5 tables to Week 6 cleaning
*PowerPoint slide 3*

Week 5 loaded a CSV into a DataFrame and selected columns and rows. We use the same tools today.

We will keep the raw table, examine missing and repeated observations, justify each change, then summarise and plot.




## The issued table and its measurements
*PowerPoint slide 4*

engineering_data.csv contains record_id, equipment_id, load_pct, ambient_c and target. One row is one observation.

record_id identifies the observation. equipment_id identifies its source and can repeat legitimately. load_pct is a percentage; ambient_c is degrees Celsius.




## Loading the supplied CSV
*PowerPoint slide 5*

Extract the student ZIP. In Colab, upload engineering_data.csv using the Files panel. Keep it beside the notebook locally.

pd.read_csv reads a CSV into a DataFrame. head() previews rows, shape gives dimensions and dtypes shows how columns were interpreted.




## Inspect the issued table
*PowerPoint slide 6*



The target column uses your programme’s stated unit.


In [ ]:
import pandas as pd
raw_table = pd.read_csv("engineering_data.csv")
print(raw_table.head())
print(raw_table.shape)
print(raw_table.dtypes)


## A small table we can check by hand
*PowerPoint slide 7*

Our practice table records room temperatures in degrees Celsius. It contains one repeated observation, one missing reading and one unusual reading.

We will use it to understand the operations before applying the assignment’s own rules.




## Create the practice table
*PowerPoint slide 8*



Five rows; record 1 appears twice.


In [ ]:
practice = pd.DataFrame({
    "record_id": [1, 1, 2, 3, 4],
    "room": ["A", "A", "A", "B", "B"],
    "temperature_c": [20, 20, None, 24, 80]
})
print(practice)


## Missing is different from zero
*PowerPoint slide 9*

Zero is a recorded number. None, NaN and pd.NA can represent missing information in different Python or pandas structures.

Use series.isna() and series.notna() to detect missing entries in pandas. Do not use value is None as a general DataFrame missing-value test.




## Find and count missing values
*PowerPoint slide 10*



Output: [False, False, True, False, False]; count 1.


In [ ]:
gaps = practice["temperature_c"].isna()
print(gaps.tolist())
print(int(gaps.sum()))
print(practice[gaps])


## A working copy preserves the original
*PowerPoint slide 11*


See the illustration on this PowerPoint slide.

Keep the original table. Apply a stated rule to a separate working copy.


## A working copy and duplicate observations
*PowerPoint slide 12*

clean = practice.copy() creates a working table. Keep practice unchanged so the effect of cleaning can be checked.

duplicated(subset="record_id", keep="first") marks later occurrences of an observation ID. drop_duplicates removes those occurrences.

Repeated room or equipment names are expected. They do not identify duplicate observations.




## Keep the first copy of each observation
*PowerPoint slide 13*



Remaining observation IDs: [1, 2, 3, 4].


In [ ]:
duplicate = practice.duplicated("record_id")
print(duplicate.tolist())
clean = practice.drop_duplicates("record_id").copy()
print(clean["record_id"].tolist())


## Median filling adds an estimate
*PowerPoint slide 14*

Sort the retained known values: 20, 24, 80. The middle value is 24. pandas median() ignores missing values.

fillna(24) fills only missing entries. A measured zero or the unusual 80 stays unchanged.

The filled value is an estimate, not a recovered measurement. A column with no known values needs an explicit policy.




## Fill only the missing temperature
*PowerPoint slide 15*



Fill: 24.0. Values: [20.0, 24.0, 24.0, 80.0].


In [ ]:
fill = clean["temperature_c"].median()
clean["estimated"] = clean["temperature_c"].isna()
clean["temperature_c"] = clean["temperature_c"].fillna(fill)
print(fill)
print(clean["temperature_c"].tolist())
print(len(practice), len(clean))


## Filtering and missing targets
*PowerPoint slide 16*

A mask is one True/False decision per row. notna() keeps known measurements; between(low, high) includes both limits by default.

Use & to combine pandas masks and parentheses around comparisons. ~ reverses True and False in a mask. Python and does not combine entire Series.

For A6, remove duplicate IDs first. Then exclude missing targets and loads outside 0–100. Calculate the ambient median only from the rows that remain.




## A filter with two conditions
*PowerPoint slide 17*



Output: [1, 1, 3]. This filter does not remove duplicates.


In [ ]:
measured = practice["temperature_c"].notna()
moderate = practice["temperature_c"].between(15, 30)
selected = practice[measured & moderate]
print(selected["record_id"].tolist())


## Numeric text and failed conversion
*PowerPoint slide 18*

pd.Series([...]) builds one labelled column from a list of values.

pd.to_numeric converts numeric text into numbers. Its default is to raise an error for text such as "24 C".

errors="coerce" replaces unconvertible entries with a missing value. Count and inspect these failures before using a mean.

A sentinel such as -999 or zero counts as a real number unless the data documentation says it represents missing information.




## Inspect values that failed conversion
*PowerPoint slide 19*



Failed value: ["24 C"]. Count: 1.


In [ ]:
text = pd.Series(["20", "24 C", "26"])
numbers = pd.to_numeric(text, errors="coerce")
failed = text.notna() & numbers.isna()
print(text[failed].tolist())
print(int(failed.sum()))


## Summaries and an audit
*PowerPoint slide 20*

mean(), min() and max() summarise a numerical column. groupby collects rows with the same label before a calculation.

An audit counts what is actually present. It must still detect a duplicate, missing target or invalid load if the supposed cleaned data contain one.




## Summarise the cleaned practice table
*PowerPoint slide 21*



Mean: 37.0. Range: 20.0–80.0. A: 22.0; B: 52.0.


In [ ]:
values = clean["temperature_c"]
print(float(values.mean()))
print(float(values.min()), float(values.max()))
means = clean.groupby("room")["temperature_c"].mean()
print(means.to_dict())


## Choosing a chart for the question
*PowerPoint slide 22*


See the illustration on this PowerPoint slide.

This week uses histograms, scatter plots and bar charts. The other chart types are reference examples.


## Charts from DataFrame columns
*PowerPoint slide 23*

A scatter plot pairs two numerical measurements. A histogram counts observations in value ranges. A bar chart can compare group means.

matplotlib supplies figures and axes. An axis label states the quantity and unit. plot methods draw on the axes supplied through ax=.




## Draw and label the practice charts
*PowerPoint slide 24*



The horizontal axis is observation ID, not elapsed time.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
clean.plot.scatter(x="record_id", y="temperature_c", ax=ax)
ax.set_xlabel("Observation ID (not time)")
ax.set_ylabel("Temperature (degrees C)")
ax.set_title("Practice observations after cleaning")
plt.show()


## Group comparison and distribution (1/2)
*PowerPoint slide 25*



Bar: group means. Histogram: counts in value intervals.


In [ ]:
fig, ax = plt.subplots()
means.plot.bar(ax=ax)
ax.set_ylabel("Mean temperature (degrees C)")
ax.set_title("Practice room means")
fig2, ax2 = plt.subplots()
clean["temperature_c"].plot.hist(bins=3, ax=ax2)
ax2.set_xlabel("Temperature (degrees C)")
ax2.set_ylabel("Observation count")
ax2.set_title("Practice temperature distribution")
plt.show()


## Pandas inside the assignment functions
*PowerPoint slide 27*

The published function names and outputs stay the same. Inside a function, pd.DataFrame(records) gives the input records column names.

Calculate with pandas. Convert back only when returning the type requested by the task. This keeps existing submissions compatible.




## Returning the requested Python type
*PowerPoint slide 28*

table.to_dict("records") returns a list containing one dictionary per row. summary.to_dict() returns a dictionary from labels to values.

table.to_numpy().tolist() returns rows of values in the selected column order. series.tolist() returns a column as a list.

Use int(...) for counts and float(...) for numerical summaries when constructing a returned dictionary.




## A table inside a function
*PowerPoint slide 29*



Output: {"A": 5}. This is a practice example.


In [ ]:
import pandas as pd
def totals(records):
    table = pd.DataFrame(records)
    result = table.groupby("site")["hours"].sum()
    return result.to_dict()
example = [{"site": "A", "hours": 2},
           {"site": "A", "hours": 3}]
print(totals(example))


## Missing values at the submission boundary
*PowerPoint slide 30*

The issued records use None for missing numeric fields. A DataFrame may represent the same gaps as NaN.

The supplied loader and records_from_table helper handle that conversion. Use them as provided; do your analysis with pandas.

The assignment still needs your cleaning, summary and audit functions. The loader is not a solution.




## Continue with your issued assignment
Open the assignment notebook named in 00_START_HERE.txt. Keep required function names, return types and units. guided_practice.ipynb is not a graded upload. Use SUBMIT_HERE.txt for the correct submission link.
